### 读取Oregano数据

In [1]:
import math
import os

In [2]:
import gravis as gv  # for visualization of the KG schema and subgraphs, developed by the author of this notebook
import igraph as ig
import pandas as pd

In [3]:
import shared_bmkg

In [4]:
project_name = "oregano"
download_dir = os.path.join(project_name, "downloads")
results_dir = os.path.join(project_name, "results")


In [5]:
download_specification = [
    ("OREGANO_V2.1.tsv", "https://figshare.com/ndownloader/files/42612700", "3e534cadf7717c705cbd5e6dd8c392a6"),

    ("ACTIVITY.tsv", "https://figshare.com/ndownloader/files/42612697", "d1faec938bd32ec9bb45ae5bba14d7a2"),
    ("COMPOUND.tsv", "https://figshare.com/ndownloader/files/42612676", "d86277d0e849c3774acb3c6be1878b44"),
    ("DISEASES.tsv", "https://figshare.com/ndownloader/files/42612685", "747a48c9786fd50912bb8d7f33d630df"),
    ("EFFECT.tsv", "https://figshare.com/ndownloader/files/42612694", "9b1656a45df7d0369dc2a8e5f52b5c2b"),
    ("GENES.tsv", "https://figshare.com/ndownloader/files/42612673", "8fe9a917b23ab715a268feda50b4a8d7"),
    ("INDICATION.tsv", "https://figshare.com/ndownloader/files/42612691", "eae2cbaebc64b1387cce1755a386d1cb"),
    ("PATHWAYS.tsv", "https://figshare.com/ndownloader/files/42612688", "d3f0abefe355c4b9cd1eea4cc8d58021"),
    ("PHENOTYPES.tsv", "https://figshare.com/ndownloader/files/42612679", "cada3e9316ba4ec654dd2b9ec2afa23c"),
    ("SIDE_EFFECT.tsv", "https://figshare.com/ndownloader/files/42612670", "8388b5a37db250128e50571d42d96cbc"),
    ("TARGET.tsv", "https://figshare.com/ndownloader/files/42612682", "46e189c556b2b010952d4212b8779cef"),

    ("oreganov2.1_metadata_complete.ttl", "https://figshare.com/ndownloader/files/42700234", "b087300f5e597d263a03c02eaae53bc7"),
]

In [6]:
df_kg = shared_bmkg.read_tsv_file(os.path.join(download_dir, "OREGANO_V2.1.tsv"), header=None)
df_kg.columns = ["subject", "predicate", "object"]

In [7]:
df_kg.tail()

,subject,predicate,object
823000,PROTEIN:22090,gene_product_of,GENE:32165
823001,PROTEIN:22091,gene_product_of,GENE:33606
823002,PROTEIN:22093,gene_product_of,GENE:35418
823003,PROTEIN:22095,gene_product_of,GENE:31330
823004,PROTEIN:22096,gene_product_of,GENE:25313


In [8]:
df_activity = shared_bmkg.read_tsv_file(os.path.join(download_dir, "ACTIVITY.tsv"))
df_activity.columns.values[0] = "id"

df_compound = shared_bmkg.read_tsv_file(os.path.join(download_dir, "COMPOUND.tsv"))
df_compound.columns.values[0] = "id"

df_diseases = shared_bmkg.read_tsv_file(os.path.join(download_dir, "DISEASES.tsv"))
df_diseases.columns.values[0] = "id"

df_effect = shared_bmkg.read_tsv_file(os.path.join(download_dir, "EFFECT.tsv"))
df_effect.columns.values[0] = "id"

df_genes = shared_bmkg.read_tsv_file(os.path.join(download_dir, "GENES.tsv"))
df_genes.columns.values[0] = "id"

df_indication = shared_bmkg.read_tsv_file(os.path.join(download_dir, "INDICATION.tsv"))
df_indication.columns.values[0] = "id"

df_pathways = shared_bmkg.read_tsv_file(os.path.join(download_dir, "PATHWAYS.tsv"))
df_pathways.columns.values[0] = "id"

df_phenotypes = shared_bmkg.read_tsv_file(os.path.join(download_dir, "PHENOTYPES.tsv"))
df_phenotypes.columns.values[0] = "id"

df_sideeffect = shared_bmkg.read_tsv_file(os.path.join(download_dir, "SIDE_EFFECT.tsv"))
df_sideeffect.columns.values[0] = "id"

df_target = shared_bmkg.read_tsv_file(os.path.join(download_dir, "TARGET.tsv"))
df_target.columns.values[0] = "id"

In [9]:
# rdf_kg = shared_bmkg.read_ttl_file(os.path.join(download_dir, "oreganov2.1_metadata_complete.ttl"))

### Attempt to identify the possible causes of a disease (e.g., genes, proteins, etc.)

A disease ID corresponds to different names across various databases.

In [10]:
df_diseases.columns

Index(['id', 'PHARMGKB', 'MESH', ' SNOMEDCT', ' UMLS', ' NDFRT', ' MEDDRA',
       'SNOMEDCT', 'NDFRT', ' MESH', 'MEDDRA', 'UMLS', 'ORPHANET', 'ICD-11',
       'ICD-10', 'OMIM', 'GARD'],
      dtype='object')

In [11]:
print(len(df_diseases))
df_diseases.head()

18333


,id,PHARMGKB,MESH,SNOMEDCT,UMLS,NDFRT,MEDDRA,SNOMEDCT,NDFRT,MESH,MEDDRA,UMLS,ORPHANET,ICD-11,ICD-10,OMIM,GARD
0,DISEASE:1,PA165108557,D054868,4325000,C0795841,N0000181176,NaN,NaN,NaN,NaN,NaN,C0795841,2308,LD44.B0,Q93.5,NaN,307
1,DISEASE:2,PA446766,D018784,75100008,C0243001,N0000003856,10060921,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,DISEASE:3,PA165109087,D018221,400153009,C0206646,N0000003695,10059354,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,DISEASE:4,PA446220,D015746,207205003;21522001,C0000737,N0000003309,10000039,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,DISEASE:5,PA165109033,D020434,398963001,C0271355,N0000004148,10053662,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
df_disease_valid = df_diseases[['id', 'ICD-10']][df_diseases['ICD-10'].notna()]

In [13]:
print(len(df_disease_valid))
df_disease_valid.head()

7617


,id,ICD-10
0,DISEASE:1,Q93.5
5,DISEASE:6,E78.6
15,DISEASE:16,B60.1+;H19.2*
22,DISEASE:23,Q77.4
33,DISEASE:34,F80.3


In [14]:
disease1 = df_disease_valid.iloc[0]['id']
print(disease1)

DISEASE:1


In [15]:
df_kg.head()

,subject,predicate,object
0,COMPOUND:1,has_code,B01AE02
1,COMPOUND:2,has_code,L01FE01
2,COMPOUND:3,has_code,R05CB13
3,COMPOUND:4,has_code,L01XX29
4,COMPOUND:5,has_code,L04AB01


In [16]:
disease_gene = df_kg[df_kg["predicate"]=="causes_condition"]
disease_gene = disease_gene[disease_gene["object"].isin(df_disease_valid["id"])]
print("num:", len(disease_gene))
disease_gene.head()

num: 5977


,subject,predicate,object
346840,GENE:26358,causes_condition,DISEASE:7531
346844,GENE:26563,causes_condition,DISEASE:7148
346845,GENE:26633,causes_condition,DISEASE:3120
346849,GENE:26949,causes_condition,DISEASE:5893
346857,GENE:29026,causes_condition,DISEASE:7140


In [17]:
disease_gene[disease_gene["object"] == disease1]

,subject,predicate,object
352958,GENE:27214,causes_condition,DISEASE:1


In [18]:
gene_protein = df_kg[df_kg["predicate"]=="gene_product_of"]
gene_protein = gene_protein[gene_protein["object"].isin(disease_gene["subject"])]
print("num:", len(gene_protein))
gene_protein.head()

num: 2576


,subject,predicate,object
812125,PROTEIN:6194,gene_product_of,GENE:33567
812129,PROTEIN:6882,gene_product_of,GENE:27501
812130,PROTEIN:6883,gene_product_of,GENE:27378
812132,PROTEIN:6885,gene_product_of,GENE:24846
812139,PROTEIN:4328,gene_product_of,GENE:33274


A disease is caused by a specific gene, and this gene produces certain proteins. Therefore, the disease is caused by these proteins.

Given a disease, it should be possible to infer its associated proteins.

In [19]:
def abductive_reasoning(disease_id, disease_gene, gene_protein):
    genes_proteins = []
    df_genes = disease_gene[disease_gene["object"] == disease_id]
    if len(df_genes) == 0:
        return genes_proteins
    for gene in df_genes["subject"]:
        df_proteins = gene_protein[gene_protein["object"] == gene]
        if len(df_proteins) == 0:
            continue
        genes_proteins.append((gene, list(df_proteins["subject"])))
    return genes_proteins



ICD-10 to ICD-9
https://www.nber.org/research/data/icd-9-cm-and-icd-10-cm-and-icd-10-pcs-crosswalk-or-general-equivalence-mappings

In [20]:
icd_mapping = pd.read_csv("icd10cmtoicd9gem.csv")
icd_mapping = icd_mapping[['icd10cm', 'icd9cm']]
icd_mapping.set_index('icd10cm', inplace=True)
icd_mapping = icd_mapping['icd9cm'].to_dict()

In [21]:
str(df_disease_valid['ICD-10'].iloc[0]).replace('.', '') in icd_mapping

True

In [22]:
df_disease_valid

,id,ICD-10
0,DISEASE:1,Q93.5
5,DISEASE:6,E78.6
15,DISEASE:16,B60.1+;H19.2*
22,DISEASE:23,Q77.4
33,DISEASE:34,F80.3
...,...,...
13700,DISEASE:13701,E72.1
13701,DISEASE:13702,D81.8
13705,DISEASE:13706,D76.1
13706,DISEASE:13707,D56.4


In [23]:
for k in icd_mapping.keys():
    print(k)

A000
A001
A009
A0100
A0101
A0102
A0103
A0104
A0105
A0109
A011
A012
A013
A014
A020
A021
A0220
A0221
A0222
A0223
A0224
A0225
A0229
A028
A029
A030
A031
A032
A033
A038
A039
A040
A041
A042
A043
A044
A045
A046
A047
A048
A049
A050
A051
A052
A053
A054
A055
A058
A059
A060
A061
A062
A063
A064
A065
A066
A067
A0681
A0682
A0689
A069
A070
A071
A072
A073
A074
A078
A079
A080
A0811
A0819
A082
A0831
A0832
A0839
A084
A088
A09
A150
A154
A155
A156
A157
A158
A159
A170
A171
A1781
A1782
A1783
A1789
A179
A1801
A1802
A1803
A1809
A1810
A1811
A1812
A1813
A1814
A1815
A1816
A1817
A1818
A182
A1831
A1832
A1839
A184
A1850
A1851
A1852
A1853
A1854
A1859
A186
A187
A1881
A1882
A1883
A1884
A1885
A1889
A190
A191
A192
A198
A199
A200
A201
A202
A203
A207
A208
A209
A210
A211
A212
A213
A217
A218
A219
A220
A221
A222
A227
A228
A229
A230
A231
A232
A233
A238
A239
A240
A241
A242
A243
A249
A250
A251
A259
A260
A267
A268
A269
A270
A2781
A2789
A279
A280
A281
A282
A288
A289
A300
A301
A302
A303
A304
A305
A308
A309
A310
A311
A312
A318
A319


In [24]:
from collections import defaultdict
diseases_reasons = defaultdict(list)
for disease_id, icd10 in df_disease_valid[['id', 'ICD-10']].values:
# for disease_id in df_disease_valid['id']:
    reasons = abductive_reasoning(disease_id, disease_gene, gene_protein)
    if len(reasons) == 0:
        continue
    
    icd10 = str(icd10).replace('.', '')
    if icd10 in icd_mapping:
        icd9 = icd_mapping[icd10]
    else:
        print("not found:", icd10)
        continue

    diseases_reasons[icd9].extend(reasons)

    

not found: F803
not found: C740
not found: E752
not found: E702
not found: E880
not found: G122
not found: E345
not found: D610
not found: D467
not found: D570;D571;D572
not found: E722
not found: Q878
not found: Q878
not found: E752
not found: D830;D839;D831;D832;D838
not found: A810
not found: E840;E841;E848
not found: E888
not found: E720
not found: Q900;Q901;Q902;Q909
not found: H508
not found: B07
not found: Q641
not found: E752
not found: D610
not found: A818
not found: E741
not found: E751
not found: A818
not found: E740
not found: M895
not found: O142
not found: E830
not found: I778
not found: D377;C254;D137
not found: I458
not found: H185
not found: H498
not found: G403
not found: Q878
not found: Q878
not found: C924
not found: C97
not found: M320;M321;M328;M329
not found: C821;C822;C823;C824;C825;C826;C827;C829;C820
not found: C831
not found: T883
not found: Q874
not found: Q878
not found: E762
not found: D448
not found: D448
not found: C900
not found: G403
not found: G711
no

In [25]:
len(diseases_reasons)

317

In [48]:
mimic_icd9 = pd.read_csv("DIAGNOSES_ICD.csv")
mimic_icd9 = set(mimic_icd9['ICD9_CODE'])

In [49]:
len(mimic_icd9)

6985

In [28]:
_diseases_reasons = {}
for k, v in diseases_reasons.items():
    if k in mimic_icd9:
        _diseases_reasons[k] = v
diseases_reasons = _diseases_reasons

In [29]:
len(diseases_reasons)

258

In [30]:
diseases_reasons

{'75839': [('GENE:27214', ['PROTEIN:6548']),
  ('GENE:32979', ['PROTEIN:12455']),
  ('GENE:33448', ['PROTEIN:21970']),
  ('GENE:26178', ['PROTEIN:2126']),
  ('GENE:29712', ['PROTEIN:19712']),
  ('GENE:33530', ['PROTEIN:6141']),
  ('GENE:33531', ['PROTEIN:16612']),
  ('GENE:25133', ['PROTEIN:11602']),
  ('GENE:31252', ['PROTEIN:10825']),
  ('GENE:25090', ['PROTEIN:10887']),
  ('GENE:25090', ['PROTEIN:10887']),
  ('GENE:35649', ['PROTEIN:1742']),
  ('GENE:35649', ['PROTEIN:1742']),
  ('GENE:26843', ['PROTEIN:10307']),
  ('GENE:30694', ['PROTEIN:15222']),
  ('GENE:25464', ['PROTEIN:271']),
  ('GENE:25464', ['PROTEIN:271']),
  ('GENE:28937', ['PROTEIN:82']),
  ('GENE:31669', ['PROTEIN:6051']),
  ('GENE:32797', ['PROTEIN:12522']),
  ('GENE:34633', ['PROTEIN:11934']),
  ('GENE:25030', ['PROTEIN:18347']),
  ('GENE:27163', ['PROTEIN:12055']),
  ('GENE:32600', ['PROTEIN:16803']),
  ('GENE:27301', ['PROTEIN:18552']),
  ('GENE:27301', ['PROTEIN:18552']),
  ('GENE:28019', ['PROTEIN:14633']),
  ('G

In [31]:
import pickle

In [32]:
pickle.dump(diseases_reasons, open(os.path.join(project_name, "diseases_reasons.pickle"), "wb"))

In [33]:
for item, _ in zip(diseases_reasons.items(), range(10)):
    print(item)
print('...')

('75839', [('GENE:27214', ['PROTEIN:6548']), ('GENE:32979', ['PROTEIN:12455']), ('GENE:33448', ['PROTEIN:21970']), ('GENE:26178', ['PROTEIN:2126']), ('GENE:29712', ['PROTEIN:19712']), ('GENE:33530', ['PROTEIN:6141']), ('GENE:33531', ['PROTEIN:16612']), ('GENE:25133', ['PROTEIN:11602']), ('GENE:31252', ['PROTEIN:10825']), ('GENE:25090', ['PROTEIN:10887']), ('GENE:25090', ['PROTEIN:10887']), ('GENE:35649', ['PROTEIN:1742']), ('GENE:35649', ['PROTEIN:1742']), ('GENE:26843', ['PROTEIN:10307']), ('GENE:30694', ['PROTEIN:15222']), ('GENE:25464', ['PROTEIN:271']), ('GENE:25464', ['PROTEIN:271']), ('GENE:28937', ['PROTEIN:82']), ('GENE:31669', ['PROTEIN:6051']), ('GENE:32797', ['PROTEIN:12522']), ('GENE:34633', ['PROTEIN:11934']), ('GENE:25030', ['PROTEIN:18347']), ('GENE:27163', ['PROTEIN:12055']), ('GENE:32600', ['PROTEIN:16803']), ('GENE:27301', ['PROTEIN:18552']), ('GENE:27301', ['PROTEIN:18552']), ('GENE:28019', ['PROTEIN:14633']), ('GENE:28022', ['PROTEIN:8647']), ('GENE:33383', ['PROTEI

In [34]:
lens = [len(v) for v in diseases_reasons.values()]
print("avg:", sum(lens)/len(lens))
print("min:", min(lens))
print("max:", max(lens))


avg: 11.15891472868217
min: 1
max: 157


In [35]:
genes = []
proteins = []
for genes_proteins in diseases_reasons.values():
    for gene, protein in genes_proteins:
        genes.append(gene)
        proteins.extend(protein)
genes = set(genes)
proteins = set(proteins)
print("genes:", len(genes))
print("proteins:", len(proteins))

genes: 1512
proteins: 1517


In [36]:
df_phenotypes_valid = df_phenotypes[df_phenotypes['ICD-10'].notna()][['id', 'ICD-10']]

In [37]:
print(len(df_phenotypes_valid['ICD-10'].unique()))
df_phenotypes_valid.head()

29


,id,ICD-10
957,PHENOTYPE:958,"R40.2 ""Coma, unspecified"""
1183,PHENOTYPE:1184,Q21.1
1187,PHENOTYPE:1188,Q21.3
1202,PHENOTYPE:1203,Q24.0
1205,PHENOTYPE:1206,Q21.1


In [38]:
diseases_reasons = pickle.load(open(os.path.join(project_name, "diseases_reasons.pickle"), "rb"))
_diseases_reasons = {}
for k, v in diseases_reasons.items():
    proteins = [protein for _, proteins in v for protein in proteins]
    _diseases_reasons[k] = proteins
pickle.dump(_diseases_reasons, open(os.path.join(project_name, "diseases_proteins_reasons.pickle"), "wb"))


In [39]:
lens = [len(v) for v in _diseases_reasons.values()]
print("avg:", sum(lens)/len(lens))
print("min:", min(lens))
print("max:", max(lens))

avg: 11.2984496124031
min: 1
max: 157


In [40]:
import json
json.dump(_diseases_reasons, open(os.path.join(project_name, "diseases_proteins_reasons.json"), "w"))

In [41]:
len(mimic_icd9)

6985

In [42]:
len(_diseases_reasons)

258

### Convert the relevant ICD-9 codes into textual descriptions.


In [43]:
from icd9_to_text import get_description

In [44]:
icd9_desc = []
for key in _diseases_reasons.keys():
    description = get_description(key)
    print(f"ICD9: {key}, Description: {description}")
    icd9_desc.append((key, description))
# save to csv
import pandas as pd
df = pd.DataFrame(icd9_desc, columns=['ICD9', 'Description'])
df.to_csv(f"{project_name}/icd9_desc.csv", index=False)

ICD9: 75839, Description: Autosomal deletions NEC
ICD9: 2725, Description: Lipoprotein deficiencies
ICD9: 7564, Description: Chondrodystrophy
ICD9: 7560, Description: Anomal skull/face bones
ICD9: 2530, Description: Acromegaly and gigantism
ICD9: 75169, Description: Biliary & liver anom NEC
ICD9: 28243, Description: Alpha thalassemia
ICD9: 74722, Description: Aortic atresia/stenosis
ICD9: 2727, Description: Lipidoses
ICD9: 3348, Description: Spinocerebellar dis NEC
ICD9: 75989, Description: Specfied cong anomal NEC
ICD9: 2871, Description: Thrombocytopathy
ICD9: 28244, Description: Beta thalassemia
ICD9: 2662, Description: B-complex defic NEC
ICD9: 75981, Description: Prader-willi syndrome
ICD9: 42789, Description: Cardiac dysrhythmias NEC
ICD9: 4530, Description: Budd-chiari syndrome
ICD9: 1123, Description: Cutaneous candidiasis
ICD9: 35989, Description: Myopathies NEC
ICD9: 75559, Description: Upper limb anomaly NEC
ICD9: 2370, Description: Unc behav neo pituitary
ICD9: 5811, Descri